## 1. Extract — Leitura das fontes de dados

O primeiro passo é entender o formato bruto de cada arquivo antes de processá-lo. Aqui exploramos como Python lida com JSON e CSV de formas diferentes.

### 1.1 Lendo o JSON (Empresa A)

Primeiro, uma leitura ingênua com `readline()` para ver o que o arquivo contém em sua forma bruta.

In [1]:
caminho_json = '../data_raw/dados_empresaA.json'

In [2]:
with open(caminho_json, 'r', encoding='utf-8') as arquivo:
    print(arquivo.readline())

{"Nome do Produto":"Blush em p\u00f3","Categoria do Produto":"Eletrodom\u00e9sticos","Pre\u00e7o do Produto (R$)":79.41,"Quantidade em Estoque":7,"Filial":"Filial 7"}



O retorno é uma string — o arquivo inteiro em uma linha. Para trabalhar com os dados de forma estruturada, precisamos do módulo `json`.

In [3]:
import json

In [4]:
with open(caminho_json, 'r', encoding='utf-8') as arquivo:
    registros_json = json.load(arquivo)

In [5]:
registros_json[0]

{'Nome do Produto': 'Blush em pó',
 'Categoria do Produto': 'Eletrodomésticos',
 'Preço do Produto (R$)': 79.41,
 'Quantidade em Estoque': 7,
 'Filial': 'Filial 7'}

In [6]:
registros_json[1]

{'Nome do Produto': 'Lápis de sobrancelha',
 'Categoria do Produto': 'Eletrodomésticos',
 'Preço do Produto (R$)': 85.47,
 'Quantidade em Estoque': 78,
 'Filial': 'Filial 8'}

Verificando os tipos retornados — `json.load` converte o arquivo em uma lista de dicionários, que é exatamente o que precisamos.

In [7]:
type(registros_json)

list

In [8]:
type(registros_json[0])

dict

### 1.2 Lendo o CSV (Empresa B)

Mesma abordagem progressiva: começamos com `readlines()` para ver o formato bruto.

In [9]:
caminho_csv = '../data_raw/dados_empresaB.csv'

In [10]:
with open(caminho_csv, 'r', encoding='utf-8') as arquivo:
    print(arquivo.readlines()[1])

Lápis de sobrancelha,Roupas,55.17,62,Filial 1,2023-04-13 18:58:06.794203



Cada linha é uma string com os valores separados por vírgula. Com `readlines()`, o cabeçalho (linha 0) e os dados ficam misturados. Vamos usar `csv.reader` para separar os campos automaticamente.

In [11]:
import csv

In [12]:
registros_csv_lista = []
with open(caminho_csv, 'r', encoding='utf-8') as arquivo:
    leitor = csv.reader(arquivo, delimiter=',')
    for linha in leitor:
        registros_csv_lista.append(linha)


In [13]:
registros_csv_lista[0][0]

'Nome do Item'

In [14]:
type(registros_csv_lista[0])

list

Com `csv.reader` cada linha vira uma lista. O problema é que para acessar `'Quantidade em Estoque'` precisaríamos saber o índice exato da coluna — frágil. Comparando o acesso entre as duas fontes:

In [15]:
# JSON: acesso por nome da chave
registros_json[0]['Quantidade em Estoque']

7

In [16]:
# CSV com reader: acesso por índice numérico (frágil)
registros_csv_lista[1][3]

'62'

A solução é usar `csv.DictReader`, que lê o cabeçalho automaticamente e retorna cada linha como um dicionário — igualando o comportamento ao JSON.

In [17]:
registros_csv = []
with open(caminho_csv, 'r', encoding='utf-8') as arquivo:
    leitor = csv.DictReader(arquivo, delimiter=',')
    for linha in leitor:
        registros_csv.append(linha)


Agora ambas as fontes têm a mesma estrutura — lista de dicionários:

In [18]:
registros_json[0]

{'Nome do Produto': 'Blush em pó',
 'Categoria do Produto': 'Eletrodomésticos',
 'Preço do Produto (R$)': 79.41,
 'Quantidade em Estoque': 7,
 'Filial': 'Filial 7'}

In [19]:
registros_csv[0]

{'Nome do Item': 'Lápis de sobrancelha',
 'Classificação do Produto': 'Roupas',
 'Valor em Reais (R$)': '55.17',
 'Quantidade em Estoque': '62',
 'Nome da Loja': 'Filial 1',
 'Data da Venda': '2023-04-13 18:58:06.794203'}

## 2. Transform — Padronização e combinação

As duas fontes têm colunas com nomes diferentes. Antes de combiná-las precisamos alinhar os nomes.

### 2.1 Inspecionando as colunas

In [20]:
colunas_json = list(registros_json[0].keys())
colunas_json

['Nome do Produto',
 'Categoria do Produto',
 'Preço do Produto (R$)',
 'Quantidade em Estoque',
 'Filial']

In [21]:
len(colunas_json)

5

In [22]:
colunas_csv = list(registros_csv[0].keys())
colunas_csv

['Nome do Item',
 'Classificação do Produto',
 'Valor em Reais (R$)',
 'Quantidade em Estoque',
 'Nome da Loja',
 'Data da Venda']

In [23]:
len(colunas_csv)

6

A Empresa A (JSON) tem 5 colunas e a Empresa B (CSV) tem 6 — a coluna extra é `'Data da Venda'`. Isso significa que, ao combinar, os registros da Empresa A ficarão sem esse campo. Vamos tratar isso mais à frente com `dict.get()`.

### 2.2 Renomeando as colunas do CSV

Criamos um dicionário de mapeamento `{nome_antigo: nome_novo}` e iteramos sobre cada registro.

In [24]:
mapeamento_colunas = {
    'Nome do Item'             : 'Nome do Produto',
    'Classificação do Produto' : 'Categoria do Produto',
    'Valor em Reais (R$)'      : 'Preço do Produto (R$)',
    'Quantidade em Estoque'    : 'Quantidade em Estoque',
    'Nome da Loja'             : 'Filial',
    'Data da Venda'            : 'Data da Venda',
}
mapeamento_colunas

{'Nome do Item': 'Nome do Produto',
 'Classificação do Produto': 'Categoria do Produto',
 'Valor em Reais (R$)': 'Preço do Produto (R$)',
 'Quantidade em Estoque': 'Quantidade em Estoque',
 'Nome da Loja': 'Filial',
 'Data da Venda': 'Data da Venda'}

In [25]:
registros_csv_padronizados = []

for registro in registros_csv:
    registro_novo = {}
    for chave_antiga, valor in registro.items():
        registro_novo[mapeamento_colunas[chave_antiga]] = valor
    registros_csv_padronizados.append(registro_novo)

# Primeiro registro após a padronização
registros_csv_padronizados[0]

{'Nome do Produto': 'Lápis de sobrancelha',
 'Categoria do Produto': 'Roupas',
 'Preço do Produto (R$)': '55.17',
 'Quantidade em Estoque': '62',
 'Filial': 'Filial 1',
 'Data da Venda': '2023-04-13 18:58:06.794203'}

Confirmando que o original não foi alterado (imutabilidade dos dados brutos):

In [26]:
registros_csv[0]

{'Nome do Item': 'Lápis de sobrancelha',
 'Classificação do Produto': 'Roupas',
 'Valor em Reais (R$)': '55.17',
 'Quantidade em Estoque': '62',
 'Nome da Loja': 'Filial 1',
 'Data da Venda': '2023-04-13 18:58:06.794203'}

### 2.3 Combinando as duas fontes

In [27]:
len(registros_json)

3123

In [28]:
len(registros_csv_padronizados)

1323

In [29]:
# Total esperado após a união
print(f'Total esperado: {len(registros_json) + len(registros_csv_padronizados)} registros')

Total esperado: 4446 registros


In [30]:
catalogo_unificado = []
catalogo_unificado.extend(registros_json)
catalogo_unificado.extend(registros_csv_padronizados)

len(catalogo_unificado)

4446

Verificando o primeiro e o último registro para garantir que a união está correta:

In [31]:
catalogo_unificado[0]   # Registro da Empresa A

{'Nome do Produto': 'Blush em pó',
 'Categoria do Produto': 'Eletrodomésticos',
 'Preço do Produto (R$)': 79.41,
 'Quantidade em Estoque': 7,
 'Filial': 'Filial 7'}

In [32]:
catalogo_unificado[-1]  # Registro da Empresa B

{'Nome do Produto': 'Sombra de olhos',
 'Categoria do Produto': 'Eletrônicos',
 'Preço do Produto (R$)': '41.73',
 'Quantidade em Estoque': '5',
 'Filial': 'Filial 6',
 'Data da Venda': '2022-11-21 18:58:06.794203'}

### 2.4 Tratando campos ausentes

Como a Empresa A não tem `'Data da Venda'`, acessar essa chave diretamente levantaria um `KeyError`. Usamos `dict.get()` com valor padrão para lidar com isso de forma segura.

In [33]:
# Acesso direto — levanta KeyError para registros da Empresa A
catalogo_unificado[0]['Data da Venda']

KeyError: 'Data da Venda'

In [34]:
# Acesso com .get() — retorna None se a chave não existir
catalogo_unificado[0].get('Data da Venda')

In [35]:
# Acesso com .get() e valor padrão — a abordagem que usaremos
catalogo_unificado[0].get('Data da Venda', 'Indisponível')

'Indisponível'

In [36]:
# Registros da Empresa B têm o campo normalmente
catalogo_unificado[-1].get('Data da Venda')

'2022-11-21 18:58:06.794203'

## 3. Load — Exportação do dataset consolidado

Com os dados combinados, o passo final é salvar em CSV. Exploramos duas abordagens aqui.

### 3.1 Tentativa com `csv.DictWriter` (e por que não funcionou)

A primeira tentativa usou `csv.DictWriter`, que escreve diretamente de dicionários. O problema: o `fieldnames` foi extraído do primeiro registro (Empresa A, sem `'Data da Venda'`), então ao tentar escrever registros da Empresa B — que têm essa coluna — o writer levantou um `ValueError`.

In [37]:
# ⚠️ Esta célula levanta ValueError — mantida para fins didáticos
colunas_empresa_a = list(catalogo_unificado[0].keys())

caminho_saida = '../data_processed/dados_combinados.csv'

with open(caminho_saida, 'w', encoding='utf-8') as arquivo:
    escritor = csv.DictWriter(arquivo, fieldnames=colunas_empresa_a)
    escritor.writeheader()
    for registro in catalogo_unificado:
        escritor.writerow(registro)

ValueError: dict contains fields not in fieldnames: 'Data da Venda'

### 3.2 Solução: `csv.writer` com tabela manual

A solução foi converter os dados para uma tabela (lista de listas) antes de escrever, usando o último registro como referência de colunas — pois ele vem da Empresa B e contém todas as 6 colunas. Campos ausentes são preenchidos com `'Indisponível'` via `dict.get()`.

In [38]:
# Usamos o último registro como referência — ele tem todas as colunas
colunas_completas = list(catalogo_unificado[-1].keys())
colunas_completas

['Nome do Produto',
 'Categoria do Produto',
 'Preço do Produto (R$)',
 'Quantidade em Estoque',
 'Filial',
 'Data da Venda']

In [39]:
tabela_exportacao = [colunas_completas]

for registro in catalogo_unificado:
    linha = [
        registro.get(coluna, 'Indisponível')
        for coluna in colunas_completas
    ]
    tabela_exportacao.append(linha)

In [40]:
tabela_exportacao[0]   # Cabeçalho

['Nome do Produto',
 'Categoria do Produto',
 'Preço do Produto (R$)',
 'Quantidade em Estoque',
 'Filial',
 'Data da Venda']

In [41]:
tabela_exportacao[1]   # Primeiro registro (Empresa A — 'Data da Venda' como 'Indisponível')

['Blush em pó', 'Eletrodomésticos', 79.41, 7, 'Filial 7', 'Indisponivel']

In [42]:
tabela_exportacao[-1]  # Último registro (Empresa B — todos os campos presentes)

['Sombra de olhos',
 'Eletrônicos',
 '41.73',
 '5',
 'Filial 6',
 '2022-11-21 18:58:06.794203']

In [43]:
caminho_saida = '../data_processed/dados_combinados.csv'

with open(caminho_saida, 'w', encoding='utf-8', newline='') as arquivo:
    escritor = csv.writer(arquivo)
    escritor.writerows(tabela_exportacao)

print(f'Arquivo exportado: {caminho_saida}')
print(f'Total de registros: {len(tabela_exportacao) - 1}')  # -1 pelo cabeçalho

Arquivo exportado: ../data_processed/dados_combinados.csv
Total de registros: 4446


---

**Pipeline concluído.** O arquivo `dados_combinados.csv` contém os 4.446 registros das duas empresas, com colunas padronizadas e campos ausentes tratados.